© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

PRE SETUP


In [ ]:
# from google.colab import drive, userdata

# drive.mount('/content/drive', force_remount=False)

# TOKEN = userdata.get('GITHUB_TOKEN')
# REPO = "MaskArchitectureAnomaly_CourseProject"

# !git clone --branch debugging_step_4 https://{TOKEN}@github.com/filoppos/MaskArchitectureAnomaly_CourseProject.git /content/project
# #!git pull origin main
# #%cd /content/project/eomt


In [ ]:
# Cella che uso io (gio) per startare mio workspace su drive
# COMMENTARE SE NON USATE
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

%cd /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# !pip install --upgrade wandb

In [ ]:
import wandb

wandb.login()

## Setup

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

TASK = "coco"  # "cityscapes" oppure "coco"

device = 0  # TODO: change to the GPU you want to use
img_idx = 0  # TODO: change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project"  # TODO: change to the dataset directory

config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Carica config COCO solo per i parametri del modello
coco_config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
with open(coco_config_path, "r") as f:

    coco_config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [ ]:
data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=2,
    num_workers=2,
    check_empty_targets=False,
    **data_module_kwargs
).setup()


## Load model

In [ ]:

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

active_config = config if TASK == "cityscapes" else coco_config

# Istanzia un data module temporaneo solo per leggere i parametri del modello COCO
if TASK == "coco":
    coco_data_module_name, coco_class_name = coco_config["data"]["class_path"].rsplit(".", 1)
    coco_data_cls = getattr(importlib.import_module(coco_data_module_name), coco_class_name)
    coco_data_kwargs = coco_config["data"].get("init_args", {})
    model_data = coco_data_cls(path=data_path, batch_size=1, num_workers=0,
                               check_empty_targets=False, **coco_data_kwargs)
else:
    model_data = data  # Cityscapes, già caricato

# Ora usi model_data.num_classes e model_data.img_size come faceva il prof
encoder_cfg = active_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=model_data.img_size, **encoder_cfg.get("init_args", {}))

network_cfg = active_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=model_data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = active_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in active_config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in active_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = active_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=model_data.img_size,
        num_classes=model_data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)
print(f"Modello {TASK}: img_size={model_data.img_size}, num_classes={model_data.num_classes}")

## Load pre-trained weights from Hugging Face Hub (Dal drive invece di huggingFace c'è li hanno dati loro)
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
WEIGHTS = {
    "cityscapes": "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/CourseProjectAnomaly/eomt_cityscapes.bin",
    "coco":       "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/CourseProjectAnomaly/eomt_coco.bin",
}

state_dict = torch.load(WEIGHTS[TASK], map_location=f"cuda:{device}", weights_only=True)
model.load_state_dict(state_dict, strict=False)
print(f"Pesi {TASK} caricati!")

##Mapping COCO --> CityScapes

Cityscapes 19 classi (tutte urban driving):
road, sidewalk, building, wall, fence, pole, traffic light, traffic sign, vegetation, terrain, sky, person, rider, car, truck, bus, train, motorcycle, bicycle
Cityscapes train IDs: road=0, sidewalk=1, building=2, wall=3, fence=4, pole=5, traffic light=6, traffic sign=7, vegetation=8, terrain=9,
sky=10, person=11, rider=12, car=13, truck=14, bus=15, train=16, motorcycle=17, bicycle=18

In [ ]:
#Per vedere i nomi delle classi COCO con i loro indici
#import json
##Scarica la lista ufficiale
#!wget -q https://raw.githubusercontent.com/cocodataset/panopticapi/master/panoptic_coco_categories.json -O /tmp/coco_cats.json
#
#with open("/tmp/coco_cats.json") as f:
#    coco_cats = json.load(f)
#
#for i, cat in enumerate(coco_cats):
#    print(f"idx={i:3d}  id={cat['id']:3d}  supercategory={cat['supercategory']:15s}  name={cat['name']}")

In [ ]:
# ── COCO panoptic idx (0-based) → Cityscapes train ID ──
# Cityscapes: road=0, sidewalk=1, building=2, wall=3, fence=4,
# pole=5, traffic light=6, traffic sign=7, vegetation=8, terrain=9,
# sky=10, person=11, rider=12, car=13, truck=14, bus=15, train=16,
# motorcycle=17, bicycle=18

COCO_TO_CITYSCAPES = {
    # THINGS (0-79)
    0:  11,  # person
    1:  18,  # bicycle
    2:  13,  # car
    3:  17,  # motorcycle
    5:  15,  # bus
    6:  16,  # train
    7:  14,  # truck
    9:   6,  # traffic light
    11:  7,  # stop sign → traffic sign
    #12:  7,   # parking meter → traffic sign (approssimazione)
    #74:   7,   # clock → traffic sign (approssimazione)



    # STUFF (80-132)
    100:  0,  # road
    123:  1,  # pavement-merged → sidewalk
    129:  2,  # building-other-merged
    91:   2,  # house → building
    82:   2,  # bridge → building (approssimazione)
    101:  2,  # roof → building
    107:  2,  # tent → building
    86:   2,   # door-stuff → building
    118:  2,   # ceiling-merged → building
    115:  2,   # window-other → building
    114:  2,   # window-blind → building
    109:  3,  # wall-brick → wall
    110:  3,  # wall-stone → wall
    111:  3,  # wall-tile → wall
    112:  3,  # wall-wood → wall
    131:  3,  # wall-other-merged → wall
    117:  4,  # fence-merged → fence
    94:   4,   # net → fence (approssimazione)
    116:  8,  # tree-merged → vegetation
    88:   8,  # flower → vegetation
    90:   9,  # gravel → terrain
    96:   9,  # platform → terrain
    97:   9,  # playingfield → terrain
    98:   9,  # railroad → terrain
    102:  9,  # sand → terrain
    126:  9,  # dirt-merged → terrain
    125:  9,  # grass-merged → terrain (era 8)
    130: 9,   # rock-merged → terrain
    105: 9,   # snow → terrain
    119: 10,  # sky-other-merged → sky
    #124: 10,  # mountain-merged → sky (approssimazione, sfondo lontano)
    #103: 10,  # sea → sky (orizzonte lontano)
    #98:  16,  # railroad → train (invece di terrain)
    #96:  16,  # platform → train (invece di terrain)

    92: 5,   # light → pole
    10: 5,   # fire hydrant → pole
    #13: 5,   # bench → pole (tentativo)

}

VOID = 255  # tutto il resto → ignorato nella valutazione

In [ ]:

def map_coco_to_cityscapes(pred_coco):
    """
    pred_coco: np.array H×W con indici COCO (0-132)
    ritorna: np.array H×W con indici Cityscapes (0-18) o 255 (void)
    """
    out = np.full_like(pred_coco, VOID)
    for coco_idx, city_idx in COCO_TO_CITYSCAPES.items():
        out[pred_coco == coco_idx] = city_idx
    return out

# def coco_logits_to_cityscapes_pred(logits_coco, num_cs=19):
#     """
#     logits_coco: [133, H, W] - pseudo-score (sigmoid * softmax) già normalizzati
#                  internamente da to_per_pixel_logits_semantic. NON applicare softmax.
#     Ritorna: np.array [H, W] con classi 0-18.
#     """
#     scores_cs = torch.zeros(
#         num_cs, *logits_coco.shape[1:],
#         device=logits_coco.device, dtype=logits_coco.dtype,
#     )
#     for coco_idx, cs_idx in COCO_TO_CITYSCAPES.items():
#         scores_cs[cs_idx] += logits_coco[coco_idx]
#     pred = scores_cs.argmax(dim=0)
#     return pred.cpu().numpy()

##  DA QUA IN GIU (nella cella) CAMBIATO ##
def coco_logits_to_cityscapes_logits(logits_coco, num_cs=19):
    """
    logits_coco: [133, H, W] pseudo-score di EoMT (già normalizzati internamente).
    Ritorna: tensor [19, H, W] aggregato sullo spazio Cityscapes (NON argmax).
    """
    scores_cs = torch.zeros(
        num_cs, *logits_coco.shape[1:],
        device=logits_coco.device, dtype=logits_coco.dtype,
    )
    for coco_idx, cs_idx in COCO_TO_CITYSCAPES.items():
        scores_cs[cs_idx] += logits_coco[coco_idx]
    return scores_cs

def coco_logits_to_cityscapes_pred(logits_coco, num_cs=19):
    """Versione che restituisce direttamente l'argmax (per compatibilità)."""
    scores_cs = coco_logits_to_cityscapes_logits(logits_coco, num_cs)
    return scores_cs.argmax(dim=0).cpu().numpy()

In [ ]:
CITYSCAPES_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
    'motorcycle', 'bicycle'
]  # cosi capiamo i dettagli per classe e dove va male

def compute_miou(pred, target, num_classes=19, ignore_index=255, return_per_class=False):
    ious = []
    per_class = {}
    for cls in range(num_classes):
        pred_mask   = (pred == cls)
        target_mask = (target == cls)
        valid_mask  = (target != ignore_index)

        intersection = (pred_mask & target_mask & valid_mask).sum()
        union        = ((pred_mask | target_mask) & valid_mask).sum()

        if union == 0:
            continue
        iou = intersection / union
        ious.append(iou)
        per_class[CITYSCAPES_NAMES[cls]] = iou

    miou = np.mean(ious) if ious else 0.0
    if return_per_class:
        return miou, per_class
    return miou

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


img, target = data.val_dataloader().dataset[img_idx]
pred_array, target_array = infer_semantic(img, target)
plot_semantic_results(img, pred_array, target_array)

## Panoptic inference (segmentation with instance IDs)

> This inference method also works when applied to a model trained for instance segmentation.

Panoptic inference assigns each pixel $[h, w]$ to the query $i$ that maximizes the product of class and mask confidence:

$$
p_i(c_i) \cdot m_i[h, w]
$$

where $c_i = \arg\max_c \, p_i(c)$ is the most likely class for query $i$. A pixel is assigned to a query only if both the class confidence and mask confidence are high. Pixels assigned to the same query form a segment labeled with $c_i$. "Stuff" segments with the same class are merged; "thing" segments are kept distinct using the query index. Low-confidence and heavily occluded predictions are filtered out.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
def infer_panoptic(img, target):
    torch.cuda.empty_cache()  #
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    return sem_pred, inst_pred, sem_target, inst_target


def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


img, target = data.val_dataloader().dataset[img_idx]
sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)

#Compute mIoU
After the visualizations

In [ ]:
## QUESTA CELLA PRIMA NON C'ERA ##
# usata per coco al posto di infer_panoptic
def infer_semantic_from_coco(img, tta_hflip=False):
    """
    Inferenza del modello COCO sul validation Cityscapes,
    con sliding window + (opzionale) TTA flip orizzontale.
    Ritorna: np.array [H, W] con classi Cityscapes 0-18.
    """
    def _forward(x):
        # x: tensor [3, H, W] sul device
        imgs = [x]
        img_sizes = [x.shape[-2:]]
        crops, origins = model.window_imgs_semantic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits_coco = model.revert_window_logits_semantic(
            crop_logits, origins, img_sizes
        )[0]  # [133, H, W]
        return coco_logits_to_cityscapes_logits(logits_coco)  # [19, H, W]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        x = img.to(device)
        logits_cs = _forward(x)
        if tta_hflip:
            logits_flip = _forward(torch.flip(x, dims=[-1]))
            logits_flip = torch.flip(logits_flip, dims=[-1])
            logits_cs = (logits_cs + logits_flip) / 2

    return logits_cs.argmax(dim=0).cpu().numpy()

In [ ]:
from tqdm import tqdm

def evaluate_and_print(model, dataloader, is_coco=False):
    intersections = np.zeros(19)
    unions = np.zeros(19)

    for batch in tqdm(dataloader):
        img, target = batch
        if isinstance(img, list): img = img[0]
        if isinstance(target, list): target = target[0]

        if is_coco:
            pred_array = infer_semantic_from_coco(img)
            target_array = model.to_per_pixel_targets_semantic(
                [target], IGNORE_INDEX)[0].numpy()
        else:
            pred_array, target_array = infer_semantic(img, target)

        valid_mask = (target_array != 255)

        # IoU accumulators
        for cls in range(19):
            pred_mask   = (pred_array == cls)
            target_mask = (target_array == cls)
            intersections[cls] += (pred_mask & target_mask & valid_mask).sum()
            unions[cls]        += ((pred_mask | target_mask) & valid_mask).sum()

    # ─── stampa IoU per classe ───
    print(f"\n{'Classe':<20} {'IoU':>6}")
    print("-" * 28)
    ious = []
    for cls in range(19):
        if unions[cls] == 0:
            continue
        iou = intersections[cls] / unions[cls]
        ious.append(iou)
        print(f"{CITYSCAPES_NAMES[cls]:<20} {iou:.4f}")

    miou = np.mean(ious)
    print("-" * 28)
    print(f"{'mIoU':<20} {miou:.4f}")

    return miou

#evaluate_and_print(model, data.val_dataloader(), is_coco=(TASK == "coco"))

# **Setup Fine-Tuning**

A) Creating the model with the Cityscapes configuration

In [ ]:
import importlib
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp.grad_scaler import GradScaler   # AMP (mixed precision)

# Importante: per il fine-tuning vogliamo la config di CITYSCAPES,
# NON quella di COCO. Quindi forziamo TASK = "cityscapes" qui.
# (Nel tuo notebook, nelle celle precedenti, avevi messo TASK = "coco"
# per testare il modello COCO grezzo; ora lo cambiamo.)
TASK = "cityscapes"
active_config = config  # `config` è la config Cityscapes già caricata sopra

# `data` è il data module di Cityscapes già caricato nelle celle precedenti.
# Da qui leggiamo num_classes (=19) e img_size.
model_data = data

# --- Costruzione encoder (DINOv2 ViT-Base, identico a prima) ---
encoder_cfg = active_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(
    img_size=model_data.img_size,
    **encoder_cfg.get("init_args", {}),
)

# --- Costruzione della rete EoMT ---
# Nota il parametro num_classes=model_data.num_classes (=19 per Cityscapes).
# È QUESTO che fa nascere la class_head già nella forma corretta!
network_cfg = active_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,            # in fine-tuning lo lasciamo False
    num_classes=model_data.num_classes,   # = 19 per Cityscapes
    encoder=encoder,
    **network_kwargs,
)

# --- Costruzione del LightningModule (wrapper) ---
lit_module_name, lit_class_name = active_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {
    k: v for k, v in active_config["model"]["init_args"].items() if k != "network"
}

model = lit_cls(
    img_size=model_data.img_size,
    num_classes=model_data.num_classes,   # = 19
    network=network,
    **model_kwargs,
).to(device)

# Importante: mettiamo il modello in MODALITÀ TRAINING (non più .eval()!)
# Questo attiva i layer come Dropout e BatchNorm in modalità training.
model.train()

print(f"✓ Modello creato con num_classes={model_data.num_classes} (Cityscapes)")
print(f"  img_size = {model_data.img_size}")

B)

In [ ]:
# --- Pesi INIZIALI (solo per la PRIMA run; in resume verranno sovrascritti) ---
# Per un rerun pulito per il report partiamo dai pesi COCO grezzi.
coco_weights_path = (
    "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/"
    "CourseProjectAnomaly/eomt_coco.bin"
)

state_dict = torch.load(coco_weights_path, map_location=f"cuda:{device}", weights_only=True)
model_state_dict = model.state_dict()

# Filtro per shape: la class_head COCO (133 classi) non combacia con Cityscapes (19),
# quindi viene scartata e re-inizializzata.
filtered_state_dict = {}
shape_mismatch, unexpected_keys = [], []
for name, param in state_dict.items():
    if name not in model_state_dict:
        unexpected_keys.append(name); continue
    if param.shape == model_state_dict[name].shape:
        filtered_state_dict[name] = param
    else:
        shape_mismatch.append(name)

load_info = model.load_state_dict(filtered_state_dict, strict=False)
print(f"\u2713 Pesi COCO caricati:           {len(filtered_state_dict)}")
print(f"\u26a0 Scartati per shape diversa:   {len(shape_mismatch)}")
print(f"\u26a0 Random (class_head etc.):     {len(load_info.missing_keys)}")


D)

In [ ]:
# ============================================================
#  CONFIG: HEAD + UPSCALE + QUERIES + BLOCCHI 10-11
#  Addestrabili: class_head, mask_head, upscale, network.q,
#                blocks.10, blocks.11, backbone.norm (+ pos_embed)
#  CONGELATI: blocchi 0-9 dell'encoder DINOv2
# ============================================================

# Quali blocchi dell'encoder sbloccare (dall'alto verso il basso).
# Per sbloccarne di piu' in futuro: aggiungi indici, es. [8, 9, 10, 11].
UNFROZEN_BLOCKS = [10, 11]

# Step 1 - congela tutto
for param in model.parameters():
    param.requires_grad = False

# Step 2 - sblocca teste + upscale + query (tutto cio' che non e' encoder)
TRAINABLE_PREFIXES = [
    "network.class_head",
    "network.mask_head",
    "network.upscale",
    "network.q",
]

trainable_param_names = []
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in TRAINABLE_PREFIXES):
        param.requires_grad = True
        trainable_param_names.append(name)

# Step 3 - sblocca gli ultimi blocchi dell'encoder + la norm finale
block_prefixes = [f"network.encoder.backbone.blocks.{b}." for b in UNFROZEN_BLOCKS]
for name, param in model.named_parameters():
    if any(name.startswith(bp) for bp in block_prefixes):
        param.requires_grad = True
        trainable_param_names.append(name)
    # la LayerNorm finale dell'encoder accompagna gli ultimi blocchi
    if name.startswith("network.encoder.backbone.norm."):
        param.requires_grad = True
        trainable_param_names.append(name)

# Step 4 - eccezione: pos_embed sbloccato in tutte le config (coerenza ablation)
encoder_exceptions = ["network.encoder.backbone.pos_embed"]
for name, param in model.named_parameters():
    if name in encoder_exceptions:
        param.requires_grad = True
        trainable_param_names.append(name)
        print(f"  Eccezione: sblocco {name} (interpolato per nuova risoluzione)")

# Report
total_params = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n=== Stato dei parametri (HEAD + UPSCALE + QUERIES + BLOCCHI {UNFROZEN_BLOCKS}) ===")
print(f"Totali:       {total_params:>12,}")
print(f"Addestrabili: {trainable_count:>12,}  ({100*trainable_count/total_params:.2f}%)")
print(f"Congelati:    {total_params-trainable_count:>12,}")
# Stampa solo un riassunto (i nomi dei blocchi sono tanti)
n_block_params = sum(1 for n in trainable_param_names if '.blocks.' in n)
print(f"\nParametri addestrabili (non-encoder + norm + pos_embed):")
for n in trainable_param_names:
    if '.blocks.' not in n:
        print(f"  - {n}")
print(f"  + {n_block_params} tensori dai blocchi {UNFROZEN_BLOCKS}")


E)

In [ ]:
# ============================================================
#  CONFIG TRAINING MULTI-RUN  (un'unica fonte di verità)
# ============================================================
CONFIG_NAME = "config-blocks-10-11"

LR_HEAD       = 1e-4    # LR per teste/upscale/query (parti nuove o leggere)
LR_BACKBONE   = 1e-5    # LR 10x piu' basso per i blocchi DINOv2 pre-addestrati
WEIGHT_DECAY  = 0.05
ETA_MIN       = 1e-6    # LR finale del cosine (per il gruppo head)

TOTAL_EPOCHS    = 10
EPOCHS_THIS_RUN = 4

BATCH_SIZE          = 2
EVAL_EVERY_N_EPOCHS = 1

import os
checkpoint_base = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project/checkpoints"
checkpoint_dir  = os.path.join(checkpoint_base, CONFIG_NAME)
os.makedirs(checkpoint_dir, exist_ok=True)
RESUME_PATH = os.path.join(checkpoint_dir, "training_state_latest.pt")

# --- Optimizer: DUE gruppi con LR diversi (discriminative learning rate) ---
# Gruppo 'head': tutto cio' che NON e' un blocco dell'encoder -> LR alto.
# Gruppo 'backbone': i blocchi DINOv2 sbloccati -> LR basso (nudge gentile).
head_params, backbone_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "encoder.backbone.blocks" in name:
        backbone_params.append(p)
    else:
        head_params.append(p)

optimizer = AdamW(
    [
        {"params": head_params,     "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ],
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)
print(f"Optimizer: head={len(head_params)} tensori @ {LR_HEAD}, "
      f"backbone={len(backbone_params)} tensori @ {LR_BACKBONE}")

# --- Scheduler: cosine su tutto il piano ---
# CosineAnnealingLR scala TUTTI i gruppi in modo proporzionale:
# head: 1e-4 -> ~1e-6 | backbone: 1e-5 -> ~1e-7. Coerente per entrambi.
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=ETA_MIN)

scaler = GradScaler()

# --- W&B: run indipendente ---
wandb.init(
    project="step-5-finetuning",
    name=CONFIG_NAME,
    id=CONFIG_NAME,
    resume="allow",
    config={
        "config_name": CONFIG_NAME,
        "trainable": "class_head + mask_head + upscale + queries + blocks 10-11 (+ pos_embed)",
        "lr_head": LR_HEAD, "lr_backbone": LR_BACKBONE,
        "weight_decay": WEIGHT_DECAY, "eta_min": ETA_MIN,
        "total_epochs": TOTAL_EPOCHS, "batch_size": BATCH_SIZE,
        "architecture": "EoMT-Base", "encoder": "DINOv2-ViT-B/14",
        "frozen_encoder": False, "unfrozen_blocks": [10, 11],
        "dataset": "Cityscapes", "num_classes": 19,
        "optimizer": "AdamW", "scheduler": "CosineAnnealingLR",
    },
)

wandb.define_metric("epoch")
wandb.define_metric("epoch/*", step_metric="epoch")
wandb.define_metric("val/*",   step_metric="epoch")

print(f"=== Setup pronto: {CONFIG_NAME} ===")
print(f"Cosine head: {LR_HEAD} -> {ETA_MIN} | backbone: {LR_BACKBONE} -> ~{ETA_MIN/10:.0e}")
print(f"Cartella checkpoint: {checkpoint_dir}")

# ============================================================
#  >>> FALLBACK OOM (decommenta SOLO se vai in out-of-memory) <<<
# ------------------------------------------------------------
# Sbloccando 2 blocchi DINOv2 le attivazioni salvate per il backward
# crescono. Se la T4 va OOM, prova in QUESTO ordine:
#
# 1) Riduci il batch size a 1 (nella cella di caricamento dataset):
#       batch_size=1
#
# 2) Gradient checkpointing sull'encoder: ricalcola le attivazioni
#    durante il backward invece di tenerle in memoria (~20-30% piu' lento,
#    ma taglia molto la RAM GPU). Decommenta una delle righe seguenti
#    (il nome del metodo dipende dall'implementazione del backbone):
#
#       model.network.encoder.backbone.set_grad_checkpointing(True)
#       # oppure, per backbone timm:
#       # model.network.encoder.backbone.grad_checkpointing = True
#
#    Verifica prima il nome corretto con:
#       # print([n for n,_ in model.named_modules() if 'backbone' in n][:5])
#
# 3) Se serve ancora memoria: batch_size=1 + grad checkpointing insieme.
# ============================================================


In [ ]:
# ============================================================
#  RESUME / INIT  — eseguire DOPO optimizer e scheduler
# ============================================================
def save_full_state(path, epochs_done, miou=None):
    torch.save({
        "model":     model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epochs_done": epochs_done,
        "miou": miou,
    }, path)

if os.path.exists(RESUME_PATH):
    print(f"\u2192 Checkpoint trovato, riprendo: {RESUME_PATH}")
    # weights_only=False perche' lo stato contiene optimizer/scheduler (non solo tensori)
    ckpt = torch.load(RESUME_PATH, map_location=f"cuda:{device}", weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])   # ripristina last_epoch del cosine
    epochs_done = ckpt["epochs_done"]
    print(f"  Epoche gia' completate: {epochs_done}/{TOTAL_EPOCHS}")
    print(f"  LR head (gruppo 0):     {optimizer.param_groups[0]['lr']:.2e}")
    if len(optimizer.param_groups) > 1:
        print(f"  LR backbone (gruppo 1): {optimizer.param_groups[1]['lr']:.2e}")
else:
    print("\u2192 Nessun checkpoint: parto dai pesi COCO (gia' caricati nella cella B)")
    epochs_done = 0


In [ ]:
from tqdm import tqdm   # barra di progresso, super utile per non impazzire
import os
import time

# Il data module Cityscapes ha bisogno che gli si dica il batch size PRIMA di
# creare i dataloader. Lo settiamo (sovrascrivendo quello iniziale).
data.batch_size = BATCH_SIZE

# Creiamo il training dataloader. Questo carica batch da batch le immagini
# di training di Cityscapes con le loro annotazioni.
train_loader = data.train_dataloader()

print(f"✓ Train dataloader pronto")
print(f"  Numero di batch per epoca: {len(train_loader)}")
print(f"  Batch size:                {BATCH_SIZE}")

# Cartella dove salvare i checkpoint durante il training
# checkpoint_dir e' gia' definito nella cella 38 (per-config subfolder).
# La definizione qui sotto e' stata RIMOSSA perche' sovrascriveva quella corretta.
# checkpoint_dir = ".../checkpoints"   # <-- causava il salvataggio nella cartella flat
# os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
def move_batch_to_device(batch, device):
    """
    Sposta ricorsivamente tutti i tensor di un batch sul device specificato.

    Il batch può essere:
    - un tensor              → lo spostiamo direttamente
    - un dict                → spostiamo ogni valore ricorsivamente
    - una lista o tupla      → spostiamo ogni elemento ricorsivamente
    - qualsiasi altra cosa   → lo lasciamo com'è (es. stringhe, numeri)
    """
    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=True)
    elif isinstance(batch, dict):
        return {k: move_batch_to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, (list, tuple)):
        moved = [move_batch_to_device(item, device) for item in batch]
        # preserviamo il tipo originale (lista o tupla)
        return type(batch)(moved)
    else:
        # non è un tensor → non c'è niente da spostare
        return batch

def train_one_epoch(model, train_loader, optimizer, scaler, epoch, num_epochs):
    """
    Esegue UNA epoca completa di training.

    Restituisce la loss media dell'epoca, utile per capire se il modello
    sta migliorando (loss che scende) o no.
    """
    # IMPORTANTE: mettiamo il modello in modalità training. Questo attiva
    # Dropout e BatchNorm in modalità "training" (con statistiche correnti).
    model.train()

    # Variabili per calcolare la loss media sull'epoca
    epoch_loss_sum = 0.0
    num_batches = 0

    # tqdm crea una barra di progresso interattiva
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True,
    )

    for batch_idx, batch in enumerate(progress_bar):

        batch = move_batch_to_device(batch, f"cuda:{device}")

        # ---- PASSO 1: Zero gradients ----
        # Azzeriamo i gradienti accumulati dall'iterazione precedente.
        # set_to_none=True è leggermente più veloce di set_to_none=False
        # (libera la memoria invece di scrivere zeri).
        optimizer.zero_grad(set_to_none=True)

        # ---- PASSO 2: Forward + calcolo loss (con AMP) ----
        # autocast() dice: "dove possibile, fai i calcoli in float16
        # invece di float32". Risultato: ~2x più veloce, ~50% di memoria
        # in meno. PyTorch decide automaticamente quali operazioni
        # convertire (le operazioni numericamente sensibili restano in float32).
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            # training_step è il metodo del LightningModule EoMT.
            # Riceve il batch, lo passa nel network, calcola la loss
            # (matching Hungarian + cross-entropy + dice + bce) e la ritorna.
            # NOTA: alcuni Lightning module ritornano un dict {"loss": tensor, ...}
            # invece di solo il tensor. Gestiamo entrambi i casi.
            output = model.training_step(batch, batch_idx)

            if isinstance(output, dict):
                loss = output["loss"]
            else:
                loss = output

        # Controllo di sanità: se la loss è NaN o inf, c'è qualcosa di rotto
        # (può succedere con AMP se il learning rate è troppo alto).
        # In quel caso saltiamo questo batch invece di rovinare il training.
        if not torch.isfinite(loss):
            print(f"\n⚠ Loss non finita al batch {batch_idx}, salto...")
            continue

        # ---- PASSO 3: Backward (con scaling per AMP) ----
        # scaler.scale(loss) moltiplica la loss per un fattore grande
        # (es. 65536) prima del backward. Questo serve perché in float16
        # i gradienti molto piccoli sarebbero 0 (underflow numerico).
        # Lo scaling li mantiene rappresentabili.
        scaler.scale(loss).backward()

        # ---- PASSO 3.5: Gradient clipping (opzionale ma utile) ----
        # Il "clipping" limita la norma dei gradienti a un valore massimo.
        # Serve a evitare aggiornamenti troppo grandi che potrebbero
        # destabilizzare il training. Best practice per transformer.
        # unscale_ rimuove lo scaling PRIMA del clipping (il clipping va
        # fatto sui gradienti veri, non su quelli scalati).
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )

        # ---- PASSO 4: Optimizer step (con scaler) ----
        # scaler.step(optimizer) fa due cose:
        #   1. Controlla se ci sono inf/NaN nei gradienti (può succedere
        #      con AMP). Se sì, salta lo step.
        #   2. Altrimenti, divide i gradienti per il fattore di scaling
        #      e chiama optimizer.step() normalmente.
        scaler.step(optimizer)

        # scaler.update() aggiusta il fattore di scaling per la prossima
        # iterazione (lo aumenta se non ci sono stati overflow, lo riduce
        # se ce ne sono stati).
        scaler.update()

        # ---- PASSO 5: Logging ----
        # Aggiorniamo le statistiche dell'epoca
        loss_value = loss.item()   # .item() converte il tensor in float
        epoch_loss_sum += loss_value
        num_batches += 1

        # ---- W&B LOGGING HERE ----
        wandb.log({
            "batch/loss": loss_value,
            "batch/lr": optimizer.param_groups[0]['lr']
        })

        # Aggiorniamo la progress bar con loss corrente e media
        progress_bar.set_postfix({
            "loss": f"{loss_value:.4f}",
            "avg_loss": f"{epoch_loss_sum / num_batches:.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    # Loss media sull'intera epoca
    avg_loss = epoch_loss_sum / max(num_batches, 1)
    return avg_loss

In [ ]:
loss_history = []
miou_history = []
start_time = time.time()

# Epoche di QUESTA sessione: da epochs_done fino al limite del piano
end_epoch = min(epochs_done + EPOCHS_THIS_RUN, TOTAL_EPOCHS)
if epochs_done >= TOTAL_EPOCHS:
    print(f"Training gia' completo ({epochs_done}/{TOTAL_EPOCHS}). Niente da fare.")
else:
    print(f"Questa sessione: epoche {epochs_done+1} \u2192 {end_epoch} (di {TOTAL_EPOCHS} totali)\n")

for epoch in range(epochs_done, end_epoch):
    absolute_epoch = epoch + 1   # 1-indexed

    # --- Training di una epoca ---
    epoch_start = time.time()
    avg_loss = train_one_epoch(
        model=model, train_loader=train_loader, optimizer=optimizer,
        scaler=scaler, epoch=epoch, num_epochs=TOTAL_EPOCHS,
    )
    epoch_time = time.time() - epoch_start
    loss_history.append(avg_loss)

    # --- Scheduler step: 1 volta per epoca, DOPO il training ---
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    elapsed = time.time() - start_time
    done_this_run = epoch - epochs_done + 1
    eta = elapsed / done_this_run * (end_epoch - epochs_done) - elapsed
    print(f"\n=== Epoch {absolute_epoch}/{TOTAL_EPOCHS} completata ===")
    print(f"  Loss media:   {avg_loss:.4f}")
    print(f"  Next LR:      {current_lr:.2e}")
    print(f"  Tempo epoca:  {epoch_time/60:.1f} min")
    print(f"  ETA sessione: {eta/60:.1f} min")

    log_dict = {"epoch/avg_loss": avg_loss, "epoch/lr": current_lr, "epoch": absolute_epoch}

    # --- Validation mIoU ---
    val_miou = None
    if absolute_epoch % EVAL_EVERY_N_EPOCHS == 0 or absolute_epoch == TOTAL_EPOCHS:
        print(f"\n--- Validation (epoch {absolute_epoch}) ---")
        model.eval()
        val_miou = evaluate_and_print(model, data.val_dataloader(), is_coco=False)
        model.train()   # CRITICO: rimetti training mode!
        miou_history.append((absolute_epoch, val_miou))
        log_dict["val/mIoU"] = val_miou
        print(f"  mIoU: {val_miou:.4f}")

    wandb.log(log_dict)

    # --- Salvataggio stato completo ---
    # 1) 'latest' per il resume (sovrascritto ogni epoca)
    save_full_state(RESUME_PATH, absolute_epoch, val_miou)
    # 2) copia per-epoca, mai sovrascritta (per confronti / report)
    epoch_ckpt = os.path.join(checkpoint_dir, f"training_state_epoch{absolute_epoch}.pt")
    save_full_state(epoch_ckpt, absolute_epoch, val_miou)
    print(f"  \u2713 Stato salvato (epoca {absolute_epoch})")

print(f"\n{'='*50}")
print(f"\u2713 Sessione completata in {(time.time()-start_time)/60:.1f} min")
if loss_history:
    print(f"  Loss: {loss_history[0]:.4f} \u2192 {loss_history[-1]:.4f}")
print(f"  Per continuare: riavvia il notebook e rilancia — riprende in automatico.")


In [ ]:
# Validation finale (solo se non gia' fatta nell'ultima epoca del loop)
if miou_history and miou_history[-1][0] == TOTAL_EPOCHS:
    print(f"mIoU finale (epoca {TOTAL_EPOCHS}): {miou_history[-1][1]:.4f}")
else:
    model.eval()
    print("\n--- mIoU finale sul Validation Set ---")
    val_miou = evaluate_and_print(model, data.val_dataloader(), is_coco=False)
    print(f"mIoU: {val_miou:.4f}")

# Chiudi la run W&B solo quando il piano e' davvero finito
if 'epochs_done' in dir() and miou_history and miou_history[-1][0] >= TOTAL_EPOCHS:
    wandb.finish()
    print("\u2713 Run W&B chiusa (training completo).")
else:
    print("Training non ancora completo: la run W&B resta aperta per la prossima sessione.")
